# Phase 4 — Phylogenetic Comparison (Kmult primary test)

Runs `src/phylo_comparison/` (Python: builds Kmult-admissible feature matrices per pattern dimension + exports the pruned tree with canonical species-key tip labels) followed by `r/phase4_kmult.R` (R: `geomorph::physignal.z` per dimension, `compare.physignal.z` across dimensions, multiple-comparisons correction) via rpy2's `%%R` cell magic, so both halves run in one notebook without switching runtimes.

**No GPU needed — use a CPU runtime.**

**Prerequisite:** Phase 3 must already have produced `reports/species_features.csv` — run `Phase3_Distance_Matrices.ipynb` first.

**IMPORTANT — read before trusting any result this notebook produces:** `r/phase4_kmult.R` has never been executed or verified against a real R installation — unlike every other piece of code in this project, it has not yet been checked against real output. Its own header comment says the same thing. Section 6 below runs a smoke test against `geomorph`'s own bundled `plethspecies` example **before** touching real data — scroll up and actually look at that output (compare it against the `geomorph` manual's own documented example) before trusting anything Section 7 produces. See README.md's Planned Approach (step 4) and the v4.1.0 changelog entry for the full design reasoning.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Open the same Drive-resident project Phase 1-3 used

Must resolve to the same `PROJECT_DIR` Phase 3 wrote `reports/species_features.csv` into.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Install Python dependencies and add the local package to the path

`biopython` is reused from Phase 3 (tree pruning); `phylo_comparison` has no further Python dependencies beyond it, numpy, and scipy.

In [ ]:
%pip install -q "biopython>=1.81"

import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import Bio, numpy, scipy
print("Deps OK - biopython", Bio.__version__, "numpy", numpy.__version__, "scipy", scipy.__version__)

## 4. Confirm Phase 3's output is actually there

In [ ]:
features_path = PROJECT_DIR / "reports" / "species_features.csv"
if not features_path.exists():
    print(f"Nothing at {features_path} yet - run Phase3_Distance_Matrices.ipynb first.")
else:
    n_lines = sum(1 for _ in open(features_path, encoding="utf-8"))
    print(f"Found {features_path} ({n_lines - 1} species row(s)).")

## 5. Run the Phase 4 Python preparation

Builds each dimension's Kmult-admissible feature matrix (Smithson-Verkuilen + logit for the three true proportions-of-count, `log1p` for everything else, then standardized) and exports the pruned tree with tip labels renamed to canonical species keys — see `src/phylo_comparison/__init__.py` and the v4.1.0 changelog entry for the full reasoning. Fails loudly (`ValueError`) if any dimension's matrix is ill-conditioned rather than silently proceeding.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", force=True
)

from phylo_comparison.config import ExportConfig, FeaturePrepConfig
from phylo_comparison.pipeline import run as run_phase4_prep

feature_prep_config = FeaturePrepConfig(species_features_csv_path=features_path)
export_config = ExportConfig(
    output_dir=PROJECT_DIR / "outputs" / "phase4",
    tree_path=PROJECT_DIR / "data" / "phylogeny" / "actinopt_12k_treePL.tre",
    species_coverage_csv_path=PROJECT_DIR / "data" / "phylogeny" / "species_coverage.csv",
)

species_order = run_phase4_prep(feature_prep_config, export_config)
print(f"\n{len(species_order)} species in the Phase 4 analysis set.")

## 6. Inspect the exported files before handing off to R

In [ ]:
phase4_dir = PROJECT_DIR / "outputs" / "phase4"
for f in sorted(phase4_dir.iterdir()):
    print(f.name, f"({f.stat().st_size} bytes)")

import csv
with open(phase4_dir / "color_kmult_features.csv", newline="", encoding="utf-8") as fh:
    rows = list(csv.reader(fh))
print("\ncolor_kmult_features.csv header + first 3 rows:")
for row in rows[:4]:
    print(row)

## 7. Install R dependencies and load the `%%R` cell magic

`rpy2` ships preinstalled on Colab; this just loads its IPython extension. `geomorph` does not ship preinstalled and compiles from source on first install — this can take **5-15 minutes**, be patient. Re-running this cell on a later session of the same runtime is a no-op if `geomorph` is already installed.

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
if (!requireNamespace("geomorph", quietly = TRUE)) {
  install.packages("geomorph", dependencies = TRUE, Ncpus = 2)
}
library(geomorph)
packageVersion("geomorph")

## 8. Run `r/phase4_kmult.R`

This sources the actual, version-controlled script — not a copy pasted into the notebook — so there is exactly one place this logic lives. Its Step 0 (smoke test against `geomorph`'s own `plethspecies` example) prints first; **scroll up and check that output against the `geomorph` manual's own documented example before trusting Steps 3-5's results on the real data**, per the script's own header comment. This is the one part of Phase 4 that could not be verified before being committed — treat everything below as provisional until you've done that check yourself.

In [ ]:
project_dir_str = str(PROJECT_DIR)

In [ ]:
%%R -i project_dir_str
setwd(project_dir_str)
source("r/phase4_kmult.R")

## 9. Read the results back into Python

A null result (non-significant, BH-corrected p) is **inconclusive, not evidence of no association** — Harmon & Glor (2010) found real power limitations even with the correct phylogenetic-signal remedy applied. See README.md.

**Not yet run in this notebook**: the secondary phylogenetically-permuted Mantel check (deliberately deferred — see the v4.1.0 changelog entry for why).

In [ ]:
import pandas as pd

results = pd.read_csv(phase4_dir / "kmult_results.csv")
results